<a href="https://colab.research.google.com/github/gracemaria321/AI-for-CyberSecurity/blob/main/LLMs_for_vul_detection_Lab1_Part_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install transformers torch datasets huggingface_hub


In [7]:
from datasets import load_dataset

# Load CodeXGLUE vulnerability detection dataset
dataset = load_dataset("code_x_glue_cc_defect_detection", split="train")

# View sample data
print(dataset[0])


{'id': 0, 'func': 'static av_cold int vdadec_init(AVCodecContext *avctx)\n\n{\n\n    VDADecoderContext *ctx = avctx->priv_data;\n\n    struct vda_context *vda_ctx = &ctx->vda_ctx;\n\n    OSStatus status;\n\n    int ret;\n\n\n\n    ctx->h264_initialized = 0;\n\n\n\n    /* init pix_fmts of codec */\n\n    if (!ff_h264_vda_decoder.pix_fmts) {\n\n        if (kCFCoreFoundationVersionNumber < kCFCoreFoundationVersionNumber10_7)\n\n            ff_h264_vda_decoder.pix_fmts = vda_pixfmts_prior_10_7;\n\n        else\n\n            ff_h264_vda_decoder.pix_fmts = vda_pixfmts;\n\n    }\n\n\n\n    /* init vda */\n\n    memset(vda_ctx, 0, sizeof(struct vda_context));\n\n    vda_ctx->width = avctx->width;\n\n    vda_ctx->height = avctx->height;\n\n    vda_ctx->format = \'avc1\';\n\n    vda_ctx->use_sync_decoding = 1;\n\n    vda_ctx->use_ref_buffer = 1;\n\n    ctx->pix_fmt = avctx->get_format(avctx, avctx->codec->pix_fmts);\n\n    switch (ctx->pix_fmt) {\n\n    case AV_PIX_FMT_UYVY422:\n\n        vda_c

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load the correct CodeBERT model for classification
MODEL_NAME = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2)  # 2 labels: Safe, Vulnerable

# Function to detect vulnerabilities
def detect_vulnerabilities(code_snippet):
    inputs = tokenizer(code_snippet, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    return {"Safe Code": probs[0][0].item(), "Vulnerable Code": probs[0][1].item()}

# ✅ **Use this for user input**
user_code = input("Enter your code snippet:\n")
result = detect_vulnerabilities(user_code)
print("\nVulnerability Scores:", result)



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Enter your code snippet:
import sqlite3, hashlib  def  login(username, password):      conn = sqlite3.connect('users.db')      query = f"SELECT * FROM users WHERE name='{username}' AND pwd='{password}'"      result = conn.execute(query).fetchone()      if result:          token = hashlib.md5(username.encode()).hexdigest()          print(f"Login OK. Token: {token}")          return token      return None

Vulnerability Scores: {'Safe Code': 0.5007500052452087, 'Vulnerable Code': 0.49925005435943604}


In [9]:
user_code = input("Enter your code snippet: ")
result = analyze_code(user_code)
print(result)



KeyboardInterrupt: Interrupted by user